# Section 1: 법률 계약서 독소조항 탐지기

실제 근로계약서에서 **불리하거나 위험한 조항**을 자동으로 탐지하고,
구조화된 분석 리포트를 생성하는 시스템을 구축합니다.

### 학습 목표
- Document Loader로 계약서를 로드
- Text Splitter로 조항별 분할
- FAISS Vector Store에 저장하고 검색
- Output Parser로 구조화된 위험도 리포트 생성

## 0. 패키지 설치 및 환경 설정

## 1. 샘플 계약서 준비

실습을 위해 **의도적으로 독소조항을 포함한 샘플 근로계약서**를 사용합니다.

## 2. 문서 로드 및 분할

## 3. Vector Store 구축

## 4. 출력 스키마 정의 (Pydantic)

독소조항 분석 결과를 구조화된 형태로 받기 위한 스키마를 정의합니다.

## 5. 독소조항 탐지 체인 구축

## 6. RAG 기반 질의응답 추가

분석 결과에 대해 **후속 질문**을 할 수 있는 RAG 체인을 추가합니다.

## 7. 전체 파이프라인 함수화

지금까지 만든 것을 재사용 가능한 함수로 정리합니다.

## 8. 출력 길이 가드 (max_tokens)

### 왜 필요한가
지금까지 `ChatUpstage(model="solar-pro", temperature=0)`에는 **출력 토큰 상한이 없습니다**.
구조화 출력(`with_structured_output`)이 스키마 준수에 실패하면 모델이 자유 텍스트로
장문을 반환할 수 있고, 이 경우 **출력 토큰이 모델 기본 상한까지 생성**되어 비용이 급증합니다.

### 권장 설정
학습 환경에서는 `max_tokens`를 명시하여 **단일 호출당 비용 상한**을 고정합니다.

| 용도 | 권장 `max_tokens` |
|------|------------------|
| 짧은 Q&A / 분류 | 256~512 |
| 구조화 출력 (Pydantic) | 1024 |
| 긴 리포트 / 요약 | 2048~4096 |

```python
llm_guarded = ChatUpstage(
    model="solar-pro",
    temperature=0,
    max_tokens=1024,   # 출력 상한 고정
    timeout=30,        # 초 단위 네트워크 타임아웃
    max_retries=2,     # 재시도 상한
)
```

> 실제 운영에서는 프롬프트 토큰 × 호출 횟수 × 재시도까지 고려해 **세션 총 비용 상한**을
> LangSmith 등으로 별도 추적하는 것이 안전합니다. Section 5에서 다룹니다.


## 정리 및 핵심 요약

### 오늘 구축한 것

```
계약서 텍스트
    |
    v
[Document + TextSplitter] --> 청크 분할
    |
    v
[Embedding + FAISS] -------> 벡터 검색 가능
    |
    v
[Prompt + LLM + Pydantic] -> 구조화된 분석 리포트
    |
    v
[RAG Q&A Chain] -----------> 후속 질의응답
```

### 새로 배운 개념

| 개념 | 역할 |
|------|------|
| Document Loader | 문서를 Document 객체로 변환 |
| RecursiveCharacterTextSplitter | 문서를 적절한 크기로 분할 |
| UpstageEmbeddings | 텍스트를 벡터로 변환 |
| FAISS | 벡터 저장 및 유사도 검색 |
| Pydantic BaseModel | 출력 스키마 정의 |
| with_structured_output() | 구조화된 출력 생성 |
| RAG Chain | 검색 결과 기반 답변 생성 |

### 다음 시간: 자율과제

배운 RAG + Output Parser를 활용하여 다른 도메인의 분석 시스템을 만들어보세요!